# 01b StatsBomb Team-Match-Metriken

Dieses Notebook implementiert BD-06. Es liest die in BD-05 gespeicherten StatsBomb Match- und Eventdaten, aggregiert Events auf Team-Match-Ebene und schreibt eine Feature-Tabelle mit genau zwei Team-Zeilen pro Spiel.

Output:

- `data/bronze/team_match_statsbomb_metrics.parquet/` als Parquet-Dataset

## Methodischer Ansatz

Die Transformation wird als Spark-DataFrame-Pipeline umgesetzt: Eingabedaten werden aus Bronze gelesen, Eventtypen werden gezielt gefiltert und Metriken per Gruppierung auf `match_id` und `team_id` aggregiert.

Das Ergebnis bleibt ein explizites Datenprodukt auf Team-Match-Ebene. Ein reales Spiel wird als zwei Beobachtungen modelliert, eine Zeile pro Team.

In [1]:
import os
import shutil

from project_paths import ensure_pyspark_path, project_root as get_project_root
from pipeline_config import DEFAULT_SPARK_MASTER, LONG_BALL_MIN_LENGTH

ensure_pyspark_path()

from pyspark.sql import SparkSession, functions as F

SPARK_MASTER = os.getenv('SPARK_MASTER', DEFAULT_SPARK_MASTER)

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
matches_path = bronze_path / 'statsbomb_matches.parquet'
events_path = bronze_path / 'statsbomb_events.parquet'
output_path = bronze_path / 'team_match_statsbomb_metrics.parquet'

spark = (
    SparkSession.builder
    .appName('football-weather-bd06-team-match-metrics')
    .master(SPARK_MASTER)
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

{
    'base_path': str(base_path),
    'matches_path': str(matches_path),
    'events_path': str(events_path),
    'output_path': str(output_path),
}


{'base_path': '/home/jovyan/work',
 'matches_path': '/home/jovyan/work/data/bronze/statsbomb_matches.parquet',
 'events_path': '/home/jovyan/work/data/bronze/statsbomb_events.parquet',
 'output_path': '/home/jovyan/work/data/bronze/team_match_statsbomb_metrics.parquet'}

## Bronze-Daten lesen

Die Inputdaten kommen ausschliesslich aus BD-05. Events liegen als Parquet-Dataset mit Part-Dateien vor; Spark kann dieses Verzeichnis direkt als DataFrame lesen.

In [2]:
matches = spark.read.parquet(str(matches_path))
events = spark.read.parquet(str(events_path))

{
    'matches': matches.count(),
    'events': events.count(),
    'match_columns': len(matches.columns),
    'event_columns': len(events.columns),
}

{'matches': 314, 'events': 89647, 'match_columns': 50, 'event_columns': 116}

## Team-Match-Basis bauen

Die Basis kommt aus der Match-Tabelle, nicht aus den Events. Dadurch erzeugt jedes Spiel deterministisch genau eine Home- und eine Away-Zeile. Eventmetriken werden danach links angehaengt.

In [3]:
common_match_columns = [
    'match_id',
    'scope_id',
    'short_name',
    'competition_id',
    'season_id',
    'competition_name',
    'statsbomb_competition_name',
    'season_name',
    'match_date',
    'kick_off',
    'match_week',
    'competition_stage_name',
    'stadium_name',
    'stadium_country_name',
]
available_common_columns = [column for column in common_match_columns if column in matches.columns]

home_rows = matches.select(
    *[F.col(column) for column in available_common_columns],
    F.col('home_team_home_team_id').cast('long').alias('team_id'),
    F.col('home_team_home_team_name').alias('team_name'),
    F.col('away_team_away_team_id').cast('long').alias('opponent_team_id'),
    F.col('away_team_away_team_name').alias('opponent_team_name'),
    F.lit(True).alias('is_home'),
    F.col('home_score').cast('long').alias('team_score'),
    F.col('away_score').cast('long').alias('opponent_score'),
)

away_rows = matches.select(
    *[F.col(column) for column in available_common_columns],
    F.col('away_team_away_team_id').cast('long').alias('team_id'),
    F.col('away_team_away_team_name').alias('team_name'),
    F.col('home_team_home_team_id').cast('long').alias('opponent_team_id'),
    F.col('home_team_home_team_name').alias('opponent_team_name'),
    F.lit(False).alias('is_home'),
    F.col('away_score').cast('long').alias('team_score'),
    F.col('home_score').cast('long').alias('opponent_score'),
)

team_match_base = home_rows.unionByName(away_rows)

team_match_base.orderBy('match_id', F.desc('is_home')).select(
    'match_id', 'team_name', 'opponent_team_name', 'is_home', 'team_score', 'opponent_score'
).show(6, truncate=False)

+--------+------------+------------------+-------+----------+--------------+
|match_id|team_name   |opponent_team_name|is_home|team_score|opponent_score|
+--------+------------+------------------+-------+----------+--------------+
|7525    |Russia      |Saudi Arabia      |true   |5         |0             |
|7525    |Saudi Arabia|Russia            |false  |0         |5             |
|7529    |Croatia     |Nigeria           |true   |2         |0             |
|7529    |Nigeria     |Croatia           |false  |0         |2             |
|7530    |France      |Australia         |true   |2         |1             |
|7530    |Australia   |France            |false  |1         |2             |
+--------+------------+------------------+-------+----------+--------------+
only showing top 6 rows



## Eventmetriken aggregieren

Definitionen fuer BD-06:

- `xg`: Summe `shot_statsbomb_xg` fuer Shot-Events.
- `shots`: Anzahl Shot-Events.
- `xg_per_shot`: `xg / shots`, bei 0 Schuessen gleich 0.
- `passes`: Anzahl Pass-Events.
- `successful_passes`: Pass-Events ohne `pass_outcome_name`, weil StatsBomb erfolgreiche Paesse als fehlenden Outcome speichert.
- `pass_completion_rate`: `successful_passes / passes`.
- `pressures`: Anzahl Pressure-Events.
- `counterpressures`: Pressure-Events mit `counterpress = true`.
- `duels`: Anzahl Duel-Events.
- `carries`: Anzahl Carry-Events.
- `long_balls`: Pass-Events mit `pass_length >= 30`.
- `long_ball_share`: `long_balls / passes`.

In [4]:
is_shot = F.col('type_name') == F.lit('Shot')
is_pass = F.col('type_name') == F.lit('Pass')
is_successful_pass = is_pass & F.col('pass_outcome_name').isNull()
is_pressure = F.col('type_name') == F.lit('Pressure')
is_counterpress = F.coalesce(F.col('counterpress').cast('boolean'), F.lit(False))
is_duel = F.col('type_name') == F.lit('Duel')
is_carry = F.col('type_name') == F.lit('Carry')
is_long_ball = is_pass & (F.col('pass_length') >= F.lit(LONG_BALL_MIN_LENGTH))

event_flags = events.select(
    F.col('match_id').cast('long').alias('match_id'),
    F.col('team_id').cast('long').alias('team_id'),
    'team_name',
    F.when(is_shot, F.coalesce(F.col('shot_statsbomb_xg').cast('double'), F.lit(0.0))).otherwise(F.lit(0.0)).alias('xg_event'),
    F.when(is_shot, F.lit(1)).otherwise(F.lit(0)).alias('shots_event'),
    F.when(is_pass, F.lit(1)).otherwise(F.lit(0)).alias('passes_event'),
    F.when(is_successful_pass, F.lit(1)).otherwise(F.lit(0)).alias('successful_passes_event'),
    F.when(is_pressure, F.lit(1)).otherwise(F.lit(0)).alias('pressures_event'),
    F.when(is_pressure & is_counterpress, F.lit(1)).otherwise(F.lit(0)).alias('counterpressures_event'),
    F.when(is_duel, F.lit(1)).otherwise(F.lit(0)).alias('duels_event'),
    F.when(is_carry, F.lit(1)).otherwise(F.lit(0)).alias('carries_event'),
    F.when(is_long_ball, F.lit(1)).otherwise(F.lit(0)).alias('long_balls_event'),
)

event_aggregates = (
    event_flags
    .groupBy('match_id', 'team_id')
    .agg(
        F.first('team_name', ignorenulls=True).alias('event_team_name'),
        F.sum('xg_event').alias('xg'),
        F.sum('shots_event').cast('long').alias('shots'),
        F.sum('passes_event').cast('long').alias('passes'),
        F.sum('successful_passes_event').cast('long').alias('successful_passes'),
        F.sum('pressures_event').cast('long').alias('pressures'),
        F.sum('counterpressures_event').cast('long').alias('counterpressures'),
        F.sum('duels_event').cast('long').alias('duels'),
        F.sum('carries_event').cast('long').alias('carries'),
        F.sum('long_balls_event').cast('long').alias('long_balls'),
    )
)

event_aggregates.orderBy('match_id', 'team_id').show(6, truncate=False)

+--------+-------+---------------+------------------+-----+------+-----------------+---------+----------------+-----+-------+----------+
|match_id|team_id|event_team_name|xg                |shots|passes|successful_passes|pressures|counterpressures|duels|carries|long_balls|
+--------+-------+---------------+------------------+-----+------+-----------------+---------+----------------+-----+-------+----------+
|7531    |779    |Argentina      |2.0805182487      |28   |796   |694              |84       |27              |19   |698    |104       |
|7531    |793    |Iceland        |1.0321244945      |9    |224   |137              |176      |15              |39   |152    |82        |
|7534    |770    |Germany        |2.2992588492      |26   |619   |521              |165      |28              |22   |527    |109       |
|7534    |794    |Mexico         |0.9932887670000001|12   |324   |247              |324      |34              |26   |290    |81        |
|7538    |790    |Sweden         |2.01512

## Metriken an Team-Match-Basis joinen und speichern

In [5]:
metric_count_columns = [
    'shots',
    'passes',
    'successful_passes',
    'pressures',
    'counterpressures',
    'duels',
    'carries',
    'long_balls',
]

team_match_metrics = (
    team_match_base
    .join(event_aggregates, on=['match_id', 'team_id'], how='left')
    .withColumn('xg', F.coalesce(F.col('xg'), F.lit(0.0)))
)

for column in metric_count_columns:
    team_match_metrics = team_match_metrics.withColumn(column, F.coalesce(F.col(column), F.lit(0)).cast('long'))

team_match_metrics = (
    team_match_metrics
    .withColumn(
        'xg_per_shot',
        F.when(F.col('shots') > 0, F.col('xg') / F.col('shots')).otherwise(F.lit(0.0)),
    )
    .withColumn(
        'pass_completion_rate',
        F.when(F.col('passes') > 0, F.col('successful_passes') / F.col('passes')).otherwise(F.lit(0.0)),
    )
    .withColumn(
        'long_ball_share',
        F.when(F.col('passes') > 0, F.col('long_balls') / F.col('passes')).otherwise(F.lit(0.0)),
    )
    .select(
        'match_id',
        'scope_id',
        'short_name',
        'competition_id',
        'season_id',
        'competition_name',
        'statsbomb_competition_name',
        'season_name',
        'match_date',
        'kick_off',
        'match_week',
        'competition_stage_name',
        'stadium_name',
        'stadium_country_name',
        'team_id',
        'team_name',
        'opponent_team_id',
        'opponent_team_name',
        'is_home',
        'team_score',
        'opponent_score',
        'xg',
        'shots',
        'xg_per_shot',
        'passes',
        'successful_passes',
        'pass_completion_rate',
        'pressures',
        'counterpressures',
        'duels',
        'carries',
        'long_balls',
        'long_ball_share',
    )
    .orderBy('match_id', F.desc('is_home'))
)

if output_path.exists():
    if output_path.is_dir():
        shutil.rmtree(output_path)
    else:
        output_path.unlink()

team_match_metrics.coalesce(1).write.mode('overwrite').parquet(str(output_path))

{
    'rows': team_match_metrics.count(),
    'matches': team_match_metrics.select('match_id').distinct().count(),
    'output_path': str(output_path),
}

{'rows': 628,
 'matches': 314,
 'output_path': '/home/jovyan/work/data/bronze/team_match_statsbomb_metrics.parquet'}

## Qualitaetschecks

Die Checks bilden die BD-06-Akzeptanzkriterien ab: Jede Spiel-Team-Kombination hat eine Zeile, jedes Spiel hat genau zwei Team-Zeilen, und alle benoetigten Metriken sind vorhanden.

In [6]:
required_metric_columns = [
    'xg',
    'shots',
    'xg_per_shot',
    'passes',
    'successful_passes',
    'pass_completion_rate',
    'pressures',
    'counterpressures',
    'duels',
    'carries',
    'long_balls',
    'long_ball_share',
]

row_count = team_match_metrics.count()
match_count = matches.select('match_id').distinct().count()
duplicate_team_match_rows = (
    team_match_metrics
    .groupBy('match_id', 'team_id')
    .count()
    .filter(F.col('count') != 1)
    .count()
)
non_two_team_matches = (
    team_match_metrics
    .groupBy('match_id')
    .count()
    .filter(F.col('count') != 2)
    .count()
)
missing_metric_values = team_match_metrics.select(
    [F.sum(F.col(column).isNull().cast('int')).alias(column) for column in required_metric_columns]
).collect()[0].asDict()

quality_checks = {
    'matches': match_count,
    'team_match_rows': row_count,
    'expected_team_match_rows': match_count * 2,
    'duplicate_team_match_rows': duplicate_team_match_rows,
    'non_two_team_matches': non_two_team_matches,
    'missing_metric_values': missing_metric_values,
}

assert quality_checks['team_match_rows'] == quality_checks['expected_team_match_rows']
assert quality_checks['duplicate_team_match_rows'] == 0
assert quality_checks['non_two_team_matches'] == 0
assert all(value == 0 for value in missing_metric_values.values())

quality_checks

{'matches': 314,
 'team_match_rows': 628,
 'expected_team_match_rows': 628,
 'duplicate_team_match_rows': 0,
 'non_two_team_matches': 0,
 'missing_metric_values': {'xg': 0,
  'shots': 0,
  'xg_per_shot': 0,
  'passes': 0,
  'successful_passes': 0,
  'pass_completion_rate': 0,
  'pressures': 0,
  'counterpressures': 0,
  'duels': 0,
  'carries': 0,
  'long_balls': 0,
  'long_ball_share': 0}}

In [7]:
team_match_metrics.select(
    'match_id',
    'team_name',
    'opponent_team_name',
    'xg',
    'shots',
    'xg_per_shot',
    'passes',
    'successful_passes',
    'pass_completion_rate',
    'pressures',
    'counterpressures',
    'duels',
    'carries',
    'long_balls',
    'long_ball_share',
).show(10, truncate=False)

+--------+------------+------------------+------------+-----+-------------------+------+-----------------+--------------------+---------+----------------+-----+-------+----------+-------------------+
|match_id|team_name   |opponent_team_name|xg          |shots|xg_per_shot        |passes|successful_passes|pass_completion_rate|pressures|counterpressures|duels|carries|long_balls|long_ball_share    |
+--------+------------+------------------+------------+-----+-------------------+------+-----------------+--------------------+---------+----------------+-----+-------+----------+-------------------+
|7525    |Russia      |Saudi Arabia      |0.0         |0    |0.0                |0     |0                |0.0                 |0        |0               |0    |0      |0         |0.0                |
|7525    |Saudi Arabia|Russia            |0.0         |0    |0.0                |0     |0                |0.0                 |0        |0               |0    |0      |0         |0.0                |


## Ergebnis

BD-06 ist erfuellt, wenn die Assertions oben erfolgreich sind und `data/bronze/team_match_statsbomb_metrics.parquet/` existiert. Die Tabelle ist auf `match_id` und `team_id` mit Wetter- und Elo-Daten joinbar.

In [8]:
spark.stop()